# System: coordinar Agents

Los mismos tres flujos, ahora con Agents como pasos.

API **2.1.3**, kernel dedicado y ejecución offline. Los actores Python son operadores deterministas.

In [1]:
from pathlib import Path
import sys
tutorial_root = next(p / "tutorials" for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "tutorials/_offline.py").is_file())
sys.path.insert(0, str(tutorial_root))
from _offline import enforce_offline, blocked_attempts
enforce_offline()
import agentic_systems as toolkit
assert toolkit.__version__ == "2.1.3", "Instala el wheel local 2.1.3: no hay fallback a 2.1.2"


In [2]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

def choose_next(data):
    return "multiply" if data["value"] >= 0 else "END"

def combine(branches):
    return {"value": sum(row["value"] for row in branches.values())}

In [3]:
for backend in ("native", "langgraph"):
    incrementer = toolkit.Agent("incrementer", increase, skills=(arithmetic,), backend=backend)
    doubler = toolkit.Agent("doubler", multiply, skills=(arithmetic,), backend=backend)
    sequential = toolkit.System("sequential", toolkit.Sequence((incrementer, doubler)), backend=backend)
    parallel = toolkit.System("parallel", toolkit.Parallel((incrementer, doubler), join=combine), backend=backend)
    decision = toolkit.System("decision", toolkit.Graph(
        start="increase", nodes=(("increase", incrementer), ("multiply", doubler)),
        edges=(("increase", choose_next), ("multiply", "END")),
    ), backend=backend)
    for system, expected in ((sequential, 8), (parallel, 10), (decision, 8)):
        result = system.run({"value": 3})
        assert result.ok and result.data == {"value": expected}
        print(backend, system.name, result.data)
    assert decision.run({"value": -3}).data == {"value": -2}

native sequential {'value': 8}
native parallel {'value': 10}
native decision {'value': 8}


langgraph sequential {'value': 8}
langgraph parallel {'value': 10}
langgraph decision {'value': 8}


In [4]:
assert blocked_attempts() == 0
print("PASS · inferencia: 0")

PASS · inferencia: 0
